In [63]:
import ipywidgets as widgets
from IPython.display import display, HTML
import numpy as np
import cv2
import time
import asyncio
import traceback 


global BOOL
BOOL = True

class DOR:
    def __init__(self, shader, res=128, fps_limit=60, size=None, quality=None):
        self.shader = shader
        self.res = res
        self.size = res  # viewport size
        if size is not None: self.size = size
        self.quality = 50
        if quality is not None: self.quality = quality
        self.target_dt = 1.0 / fps_limit
        
        self.t = 0.0
        self.running = False
        self.paused = False
        self.task = None
        
        self.img_widget = widgets.Image(format='jpeg', width=size, height=size)
        self.fps_label = widgets.Label(value="Ready")
        # Use an Output widget for detailed error messages
        self.error_output = widgets.Output() 
        
        btn_layout = widgets.Layout(width='30px', padding='0px')
        
        self.btn_play = widgets.Button(icon='pause', layout=btn_layout)
        self.btn_reset = widgets.Button(icon='refresh', layout=btn_layout)
        self.btn_stop = widgets.Button(icon='stop', button_style='danger', layout=btn_layout)
        
        self.btn_bool = widgets.Button(
          description='', 
          icon='lightbulb-o', 
          layout=btn_layout,
          tooltip='Toggle'
        )
        
        self.controls = widgets.HBox([self.btn_bool, self.btn_play, 
                                      self.btn_reset, self.btn_stop])
        self.ui = widgets.VBox([self.img_widget, 
                                self.fps_label, self.controls, 
                                self.error_output]) # Add error_output to UI
        
        self.btn_play.on_click(self.toggle_pause)
        self.btn_reset.on_click(self.reset_time)
        self.btn_stop.on_click(self.stop_render_btn)
        
        self.btn_bool.on_click(self.toggle_bool) 
        
        display(self.ui)

  
    def toggle_bool(self, _):
        global BOOL
        BOOL = not BOOL
    
    def toggle_pause(self, _):
        self.paused = not self.paused
        self.btn_play.icon = 'play' if self.paused else 'pause'
    
    def reset_time(self, _):
        self.t = 0.0
        if self.paused:
            self.render_frame()
    
    def stop_render_btn(self, _):
        self.stop()
        self.fps_label.value = "Stopped."
        self.error_output.clear_output() # Clear errors on stop
    
    def stop(self):
        self.running = False
        if self.task:
            self.task.cancel()
            self.task = None
          
    
    def show(self, img):
      """img: (c, h, w)"""
      
      img_int = (img.transpose().clip(0, 1) * 255).astype('uint8')
            
      if img_int.shape[-1] == 3: 
          img_int = cv2.cvtColor(img_int, cv2.COLOR_RGB2BGR)
      elif img_int.shape[-1] == 4: 
          img_int = cv2.cvtColor(img_int, cv2.COLOR_RGBA2BGR)
    
      _, enc_data = cv2.imencode('.jpg', img_int, 
                                 [int(cv2.IMWRITE_JPEG_QUALITY), self.quality])
      self.img_widget.value = enc_data.tobytes()
      
    def render_frame(self, t=None):
      if t: self.t = t
      img = self.shader(self.res, self.t)
    
      self.show(img)
      
    
    async def loop(self):
      self.running = True
      self.btn_stop.disabled = False
      self.error_output.clear_output() # Clear previous errors when starting new loop
      
      try:
        while self.running:
          loop_start = time.time()
          
          if not self.paused:
              self.render_frame()
              self.t += self.target_dt 
          
          dt = time.time() - loop_start
          if dt > 0:
              self.fps_label.value = f"FPS: {1.0/dt:.1f} | \
                                      t={self.t:.2f}"
          
          wait = self.target_dt - dt
          await asyncio.sleep(max(wait, 0.001))
              
      except asyncio.CancelledError: 
          pass
      except Exception as e:
          # Use the Output widget to display the traceback
          with self.error_output:
              traceback.print_exc()
          self.fps_label.value = "Error occurred. Check details below."
          
      finally:
          self.running = False
    
    def start(self):
        print(self.shader.__name__)
        if self.task: self.task.cancel()
        if self.running: return
        self.task = asyncio.create_task(self.loop())


def render(shader, res=128, fps=60):
  try:dor.stop()
  except NameError: pass
  dor = DOR(shader, res=res, fps_limit=fps)
  dor.start()

In [84]:
def shader(res, t):
  r, b = np.meshgrid(*(np.linspace(0, 1, res) for _ in (0,1)))
  g = np.zeros((res, res)) + np.sin(t)**2. 
  
  return np.stack((r,g,b), 0) * BOOL

render(shader)



shader
